## QUESTION 2

In [4]:
from copy import deepcopy

# Global counters (for report)
backtrack_calls = 0
backtrack_failures = 0



# Helper: Read Sudoku from file
def read_sudoku(filename):
    board = []
    with open(filename, 'r') as f:
        for line in f:
            row = [int(x) for x in line.strip()]
            board.append(row)
    return board



# Create domains
def create_domains(board):
    domains = {}

    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}
    return domains



# Get neighbors of a cell
def get_neighbors(cell):
    r, c = cell
    neighbors = set()

    for i in range(9):
        neighbors.add((r, i))
        neighbors.add((i, c))

    # 3x3 box
    box_r = (r // 3) * 3
    box_c = (c // 3) * 3

    for i in range(3):
        for j in range(3):
            neighbors.add((box_r + i, box_c + j))

    neighbors.remove(cell)
    return neighbors



# AC-3 Algorithm
def ac3(domains):
    queue = [(xi, xj) for xi in domains for xj in get_neighbors(xi)]

    while queue:
        xi, xj = queue.pop(0)

        if revise(domains, xi, xj):
            if len(domains[xi]) == 0:
                return False

            for xk in get_neighbors(xi):
                if xk != xj:
                    queue.append((xk, xi))
    return True



# Revise step
def revise(domains, xi, xj):
    revised = False

    for x in set(domains[xi]):
        # If no possible value in xj allows xi != xj
        if all(x == y for y in domains[xj]):
            domains[xi].remove(x)
            revised = True

    return revised



# Check if assignment complete
def is_complete(domains):
    return all(len(domains[cell]) == 1 for cell in domains)



# Select unassigned variable
# (MRV heuristic - simple)
def select_unassigned(domains):
    return min(
        [cell for cell in domains if len(domains[cell]) > 1],
        key=lambda x: len(domains[x])
    )



# Forward checking
def forward_check(domains, var, value):
    new_domains = deepcopy(domains)

    new_domains[var] = {value}

    for neighbor in get_neighbors(var):
        if value in new_domains[neighbor]:
            new_domains[neighbor].discard(value)

            if len(new_domains[neighbor]) == 0:
                return None

    return new_domains



# Backtracking
def backtrack(domains):
    global backtrack_calls, backtrack_failures

    backtrack_calls += 1

    if is_complete(domains):
        return domains

    var = select_unassigned(domains)

    for value in domains[var]:
        new_domains = forward_check(domains, var, value)

        if new_domains is not None:
            if ac3(new_domains):
                result = backtrack(new_domains)
                if result is not None:
                    return result

    backtrack_failures += 1
    return None



# Solve Sudoku
def solve_sudoku(board):
    global backtrack_calls, backtrack_failures

    backtrack_calls = 0
    backtrack_failures = 0

    domains = create_domains(board)

    # Initial AC-3
    ac3(domains)

    result = backtrack(domains)

    return result, backtrack_calls, backtrack_failures



# Print Sudoku nicely
def print_sudoku(domains):
    grid = [[0]*9 for _ in range(9)]

    for (r, c), val in domains.items():
        grid[r][c] = list(val)[0]

    for row in grid:
        print(" ".join(map(str, row)))


In [5]:
board = read_sudoku("easy.txt")
solution, calls, failures = solve_sudoku(board)

print_sudoku(solution)
print("Calls:", calls)
print("Failures:", failures)

7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3
Calls: 1
Failures: 0


In [4]:
board = read_sudoku("medium.txt")
solution, calls, failures = solve_sudoku(board)

print_sudoku(solution)
print("Calls:", calls)
print("Failures:", failures)

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9
Calls: 1
Failures: 0


In [2]:
board = read_sudoku("hard.txt")
solution, calls, failures = solve_sudoku(board)

print_sudoku(solution)
print("Calls:", calls)
print("Failures:", failures)

9 8 7 6 5 4 3 2 1
2 4 6 1 7 3 9 8 5
3 5 1 9 2 8 7 4 6
1 2 8 5 3 7 6 9 4
6 3 4 8 9 2 1 5 7
7 9 5 4 6 1 8 3 2
5 1 9 2 8 6 4 7 3
4 7 2 3 1 9 5 6 8
8 6 3 7 4 5 2 1 9
Calls: 6341
Failures: 6328


In [3]:
board = read_sudoku("veryhard.txt")
solution, calls, failures = solve_sudoku(board)

print_sudoku(solution)
print("Calls:", calls)
print("Failures:", failures)

4 6 2 8 3 1 9 5 7
7 9 5 4 2 6 1 8 3
3 8 1 7 9 5 4 2 6
1 7 3 9 8 4 2 6 5
6 5 9 3 1 2 7 4 8
2 4 8 5 6 7 3 1 9
9 2 6 1 7 8 5 3 4
8 3 4 2 5 9 6 7 1
5 1 7 6 4 3 8 9 2
Calls: 8
Failures: 1
